<a href="https://colab.research.google.com/github/Not-kh-lily-23/dbank-longitudinal-prediction/blob/main/statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install statsmodels pingouin
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import os
from google.colab import drive
drive.mount('/content/drive')
base='/content/drive/MyDrive/DementiaBank Project'
csv_path=os.path.join(base,'nlp_features_final.csv')
df=pd.read_csv(csv_path)
df=df.dropna(subset=['age_float','education'])
features=['mlu_w','mattr','pn_ratio','disfluency_rate']
results=[]
print("Running ANCOVA adjusting for Age and Education...\n")
for feat in features:
    formula=f"{feat} ~ C(cohort_status) + age_float + education"
    model=ols(formula, data=df).fit()
    anova_table=sm.stats.anova_lm(model, typ=2)
    p_val=anova_table.loc['C(cohort_status)', 'PR(>F)']
    f_val=anova_table.loc['C(cohort_status)', 'F']
    results.append({
        'Feature': feat,
        'F_Statistic': f_val,
        'Raw_p_value': p_val
    })
res_df=pd.DataFrame(results)
rejected,p_adjusted,_,_=multipletests(res_df['Raw_p_value'],alpha=0.05,method='fdr_bh')
res_df['Adjusted_p_value']=p_adjusted
res_df['Significant']=rejected
display(res_df.sort_values('Adjusted_p_value'))
stats_op=os.path.join(base,'ancova_results.csv')
res_df.to_csv(stats_op,index=False)
print(f"statistics are stored: {stats_op}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 1.7 MB/s eta 0:00:00
Mounted at /content/drive
Running ANCOVA adjusting for Age and Education...



,Feature,F_Statistic,Raw_p_value,Adjusted_p_value,Significant
0,mlu_w,3.156906,0.044098,0.100081,False
3,disfluency_rate,3.027655,0.050041,0.100081,False
2,pn_ratio,1.369673,0.255912,0.341216,False
1,mattr,0.112413,0.893716,0.893716,False


statistics are stored: /content/drive/MyDrive/DementiaBank Project/ancova_results.csv
